# GLB to OBJ Conversion with pylmesh

Convert GLB (binary GLTF) files to OBJ format with Draco decompression support.

## Import Library

In [1]:
import pylmesh
import os

print(f"pylmesh version: {pylmesh.__version__}")
print("Supports Draco decompression: Yes")

pylmesh version: 1.0.0
Supports Draco decompression: Yes


## Load GLB File

In [6]:
# Load a GLB file
glb_file = "/scratch/quantization/12345.glb"  # Change to your GLB file path

if not os.path.exists(glb_file):
    print(f"Error: File not found: {glb_file}")
else:
    print(f"File: {glb_file}")
    print(f"Size: {os.path.getsize(glb_file) / 1024:.2f} KB")
    
    # Check if Draco compressed
    with open(glb_file, 'rb') as f:
        data = f.read()
        if b'KHR_draco_mesh_compression' in data:
            print("Compression: Draco ✓")
        else:
            print("Compression: None")
    
    try:
        mesh = pylmesh.load_mesh(glb_file)
        print(f"\n✓ Loaded successfully!")
        print(f"Vertices: {mesh.vertex_count()}")
        print(f"Faces: {mesh.face_count()}")
    except Exception as e:
        print(f"\n✗ Failed to load: {e}")

File: /scratch/quantization/12345.glb
Size: 752.23 KB
Compression: Draco ✓

✓ Loaded successfully!
Vertices: 339890
Faces: 669804


## Inspect Mesh Data

In [7]:
if 'mesh' in locals():
    print("First 5 vertices:")
    for i, v in enumerate(mesh.vertices[:5]):
        print(f"  v{i}: ({v.x:.3f}, {v.y:.3f}, {v.z:.3f})")

    print(f"\nFirst 3 faces:")
    for i, f in enumerate(mesh.faces[:3]):
        print(f"  f{i}: {f.indices}")
else:
    print("No mesh loaded. Run the previous cell first.")

First 5 vertices:
  v0: (470.527, 929.878, -1014.292)
  v1: (470.449, 929.878, -1014.344)
  v2: (470.545, 929.878, -1014.379)
  v3: (470.884, 930.643, -1015.196)
  v4: (470.867, 930.556, -1015.196)

First 3 faces:
  f0: [0, 1, 2]
  f1: [3, 4, 5]
  f2: [6, 7, 8]


## Export to OBJ

In [8]:
if 'mesh' in locals():
    # Export to OBJ format
    obj_file = "/scratch/quantization/outputx.obj"
    
    try:
        pylmesh.save_mesh(obj_file, mesh)
        print(f"✓ Exported to: {obj_file}")
        print(f"File size: {os.path.getsize(obj_file) / 1024:.2f} KB")
    except Exception as e:
        print(f"✗ Export failed: {e}")
else:
    print("No mesh loaded. Run the load cell first.")

✓ Exported to: /scratch/quantization/outputx.obj
File size: 23241.62 KB


## Verify OBJ Export

In [5]:
if 'obj_file' in locals() and os.path.exists(obj_file):
    # Load the exported OBJ to verify
    mesh_obj = pylmesh.load_mesh(obj_file)
    
    print(f"Verified OBJ file:")
    print(f"Vertices: {mesh_obj.vertex_count()}")
    print(f"Faces: {mesh_obj.face_count()}")
    print(f"\nConversion successful! ✓")
else:
    print("No OBJ file to verify. Run the export cell first.")

Verified OBJ file:
Vertices: 339890
Faces: 669804

Conversion successful! ✓


## Batch Conversion: GLB to OBJ

In [ ]:
import glob

# Find all GLB files in directory
input_dir = "/scratch/quantization/"  # Change to your directory
glb_files = glob.glob(os.path.join(input_dir, "*.glb"))

print(f"Found {len(glb_files)} GLB files in {input_dir}\n")

for glb_file in glb_files[:5]:  # Limit to first 5 files
    try:
        # Load GLB
        mesh = pylmesh.load_mesh(glb_file)
        
        # Generate OBJ filename
        base = os.path.splitext(os.path.basename(glb_file))[0]
        obj_file = base + ".obj"
        
        # Export to OBJ
        pylmesh.save_mesh(obj_file, mesh)
        
        glb_size = os.path.getsize(glb_file) / 1024
        obj_size = os.path.getsize(obj_file) / 1024
        
        print(f"✓ {os.path.basename(glb_file):20} ({glb_size:8.2f} KB) -> {obj_file:20} ({obj_size:8.2f} KB)")
        print(f"  {mesh.vertex_count()}v, {mesh.face_count()}f")
    except Exception as e:
        print(f"✗ {os.path.basename(glb_file):20} -> Failed: {e}")

print("\nBatch conversion complete!")

## Export to Multiple Formats

In [ ]:
if 'mesh' in locals():
    formats = [
        ("output.obj", "OBJ"),
        ("output.stl", "STL"),
        ("output.ply", "PLY"),
        ("output.off", "OFF"),
        ("output.gltf", "GLTF")
    ]

    print("Exporting to all formats:\n")
    for filename, format_name in formats:
        try:
            pylmesh.save_mesh(filename, mesh)
            size = os.path.getsize(filename) / 1024
            print(f"✓ {format_name:6} -> {filename:15} ({size:.2f} KB)")
        except Exception as e:
            print(f"✗ {format_name:6} -> Failed: {e}")
else:
    print("No mesh loaded. Run the load cell first.")

## File Size Comparison

In [ ]:
if 'mesh' in locals():
    import matplotlib.pyplot as plt
    
    # Export to all formats
    formats = {
        "OBJ": "test.obj",
        "STL": "test.stl",
        "PLY": "test.ply",
        "OFF": "test.off",
        "GLTF": "test.gltf",
        "GLB": "test.glb"
    }
    
    sizes = {}
    for name, filename in formats.items():
        try:
            pylmesh.save_mesh(filename, mesh)
            sizes[name] = os.path.getsize(filename) / 1024
        except:
            pass
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(sizes.keys(), sizes.values())
    plt.ylabel('File Size (KB)')
    plt.title('File Size Comparison by Format')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print("\nFile sizes:")
    for name, size in sorted(sizes.items(), key=lambda x: x[1]):
        print(f"  {name:6}: {size:8.2f} KB")
else:
    print("No mesh loaded. Run the load cell first.")